
## **YOLO 26n Object Detection Model Fine-Tuning Notebook**

This notebook is used, with hyperparameters adjusted as needed, to fine-tune the object detection model.

##### Notebook Setup
* Install packages
* Import Modules
* Unzip dataset to local Colab storage
* Verify directory and file paths exist

In [ ]:
# Install packages
from ultralytics import YOLO
from IPython.display import Image, display
import os
import zipfile

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Unzip dataset to local Colab storage for faster processing
zip_path = "/content/drive/MyDrive/cow_ear_tag_detector/datasets.zip"
extract_path = "/content"

# Extract file
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)
print("Dataset extraction complete")

# Verify file location
!ls -l /content/dataset/

In [ ]:
# verify paths exist
print(os.path.exists("/content/dataset/data.yaml"))
print(os.path.exists("/content/dataset/images/train"))
print(os.path.exists("/content/dataset/labels/train"))
print(os.path.exists("/content/dataset/images/val"))
print(os.path.exists("/content/dataset/labels/val"))
print(os.path.exists("/content/drive/MyDrive/cow_ear_tag_detector/runs"))

Model Training

In [ ]:
# load pretrained model
model = YOLO("yolo26n.pt")

# train the model
results = model.train(
    data="/content/dataset/dataset.yaml",
    epochs=50,
    patience=20,
    imgsz=640,
    batch=-1,
    optimizer="auto",
    freeze=None,
    cache=True,
    save=True,
    save_period=5,
    name="run1",
    project="/content/drive/MyDrive/cow_ear_tag_detector/runs"
    )
print("Run is complete")

Display Training Results

In [ ]:
# View training curves
display(Image("/content/drive/MyDrive/cow_ear_tag_detector/runs/run1/confusion_matrix.png"))

In [ ]:
display(Image("/content/drive/MyDrive/cow_ear_tag_detector/runs/run1/results.png"))

Model Validation

In [ ]:
# Load best model
best_model = YOLO("/content/drive/MyDrive/cow_ear_tag_detector/runs/run1/weights/best.pt")

# Validate
metrics = best_model.val(
    data="/content/dataset/data.yaml",
    name="val_run1",
    project="/content/drive/MyDrive/cow_ear_tag_detector/runs")

print("Validation for run is complete")

# Results
print(f"Mean precision: {metrics.box.mp}")
print(f"Mean recall: {metrics.box.mr}")
print(f"Fitness: {metrics.box.fitness()}")
print(f"mAP50-95: {metrics.box.map}")
print(f"mAP50: {metrics.box.map50}")
print(f"mAP75: {metrics.box.map75}")

# Per-stage timing in ms per image
print(f"Timing breakdown (ms/image): {metrics.speed}")